In [1]:
!pip -q install langgraph langchain langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.2 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")

    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY not found.")

    print("✅ API Key Loaded Successfully.")

except Exception as e:
    print("❌", e)

✅ API Key Loaded Successfully.


In [3]:
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

In [4]:
@tool
def get_weather(city: str) -> str:
    """
    Returns the current temperature in Celsius for a city.
    """

    weather_data = {
        "London": 18,
        "New York": 24,
        "Tokyo": 29,
        "Delhi": 35,
        "Hyderabad": 31
    }

    temp = weather_data.get(city, 25)

    return f"The current temperature in {city} is {temp}°C."


@tool
def calculate(expression: str) -> str:
    """
    Safely evaluate a mathematical expression.
    """

    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Calculation Error: {e}"

In [5]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0
)

In [6]:
tools = [get_weather, calculate]

agent = create_react_agent(
    llm,
    tools
)

/tmp/ipykernel_1182/1243542782.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [7]:
query = """
What is the current temperature in London in Fahrenheit?

Also calculate 15% of 320 and add it to the temperature.
"""

In [8]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": query
            }
        ]
    }
)

In [9]:
print("="*80)
print("AGENT EXECUTION TRACE")
print("="*80)

for message in response["messages"]:
    print(message)
    print()

AGENT EXECUTION TRACE
content='\nWhat is the current temperature in London in Fahrenheit?\n\nAlso calculate 15% of 320 and add it to the temperature.\n' additional_kwargs={} response_metadata={} id='a9fffa9b-49ff-4830-9439-1fca6dd25cdb'

content='' additional_kwargs={'tool_calls': [{'id': '1qghgytxa', 'function': {'arguments': '{"city":"London"}', 'name': 'get_weather'}, 'type': 'function'}, {'id': 't51346gkz', 'function': {'arguments': '{"expression":"get_weather(city=\\"London\\") * 9/5 + 32 + 15 * 320 / 100"}', 'name': 'calculate'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 294, 'total_tokens': 343, 'completion_time': 0.082680285, 'completion_tokens_details': None, 'prompt_time': 0.024052531, 'prompt_tokens_details': None, 'queue_time': 0.069387462, 'total_time': 0.106732816}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': Non

In [10]:
print("="*80)
print("FINAL ANSWER")
print("="*80)

print(response["messages"][-1].content)

FINAL ANSWER
The current temperature in London is 18°C. The result of the calculation is 107.0.


In [11]:
print("="*80)
print("ANALYSIS")
print("="*80)

print("""
1. The agent received a complex multi-step question.

2. It used the weather tool to obtain the temperature.

3. It used the calculator tool to:
   • Convert Celsius to Fahrenheit.
   • Calculate 15% of 320.
   • Add the values.

4. The ReAct agent automatically decided
   which tool to call and in what order.

5. This demonstrates the reasoning and
   acting loop of a LangGraph ReAct Agent.
""")

ANALYSIS

1. The agent received a complex multi-step question.

2. It used the weather tool to obtain the temperature.

3. It used the calculator tool to:
   • Convert Celsius to Fahrenheit.
   • Calculate 15% of 320.
   • Add the values.

4. The ReAct agent automatically decided
   which tool to call and in what order.

5. This demonstrates the reasoning and
   acting loop of a LangGraph ReAct Agent.

